In [0]:
%sql
create catalog if not exists Weather_Analytics;
create schema if not exists Weather_Analytics.bronze;
create schema if not exists Weather_Analytics.silver;
create schema if not exists Weather_Analytics.gold;
create schema if not exists Weather_Analytics.Data;
create volume if not exists Weather_Analytics.Data.schema;
create volume if not exists Weather_Analytics.Data.raw_weather_data;
create volume if not exists Weather_Analytics.Data.raw_station_master;
create volume if not exists Weather_Analytics.bronze.weather_data_checkpoint;

In [0]:
%sql
CREATE TABLE if not exists Weather_Analytics.bronze.bronze_weather
TBLPROPERTIES (delta.enableChangeDataFeed = true)


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:
station_df = spark.read.csv(
    "/Volumes/Weather_Analytics/Data/raw_station_master/",
    header=True,
    inferSchema=True
)

station_df = station_df \
    .withColumn("city", F.trim(F.col("city"))) \
    .withColumn("station_id", F.trim(F.col("station_id"))) \
    .dropDuplicates(["city", "station_id"]) \
    .withColumn("ingestion_time", F.current_timestamp())

if not spark.catalog.tableExists("Weather_Analytics.bronze.bronze_station_master"):

    station_df.write \
        .format("delta") \
        .mode("overwrite") \
        .option("delta.enableChangeDataFeed", "true") \
        .saveAsTable("Weather_Analytics.bronze.bronze_station_master")

else:
    station_df.createOrReplaceTempView("station_updates")
    spark.sql("""
    MERGE INTO Weather_Analytics.bronze.bronze_station_master t
    USING station_updates s
    ON t.city = s.city AND t.station_id = s.station_id

    WHEN MATCHED THEN UPDATE SET *

    WHEN NOT MATCHED THEN INSERT *
    """)

In [0]:
weather_data = spark.readStream.format("cloudFiles") \
    .option("cloudFiles.format", "csv") \
    .option("header", "true") \
    .option("cloudFiles.schemaLocation", "/Volumes/Weather_Analytics/Data/schema") \
    .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
    .option("cloudFiles.maxFilesPerTrigger", "1") \
    .option("rescuedDataColumn", "_rescued_data") \
    .option("cloudFiles.inferColumnTypes", "false") \
    .option("escape",'"')\
    .option("quote",'"')\
    .load("/Volumes/Weather_Analytics/Data/raw_weather_data/")

In [0]:
weather_data = weather_data \
    .withColumn("ingestion_time", F.current_timestamp()) \
    .withColumn("load_date", F.current_date()) \
    .withColumn("source_file", F.col("_metadata.file_path")) \
    .withColumn("batch_id", F.col("_metadata.file_modification_time").cast("string")) \
    .withColumn("record_status", F.lit("raw"))

In [0]:
weather_data.writeStream \
    .format("delta") \
    .option("mergeSchema", "true") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/Weather_Analytics/bronze/weather_data_checkpoint") \
    .trigger(availableNow=True) \
    .toTable("Weather_Analytics.bronze.bronze_weather")

In [0]:
df = spark.read.table("Weather_Analytics.bronze.bronze_weather")
display(df)

city,country,event_time,temperature,humidity,wind_speed,pressure,weather_condition,rainfall,station_id,data_provider,uv_index,visibility,condition,feels_like_temperature,sensor_payload,_rescued_data,ingestion_time,load_date,source_file,batch_id,record_status
Chennai,India,2026-03-01 00:00:00,36.4,84,18.7,1005.4,Foggy,7.3,STN_CHN_02,AtmosAPI,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Bangalore,India,2026-03-01 00:00:00,24.3,56,11.6,904.2,Partly Cloudy,0.9,STN_BLR_02,ClimaData,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Hyderabad,India,2026-03-01 00:00:00,29.7,68,7.2,1002.8,Rainy,5.7,STN_HYD_01,SkyWatch,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Mumbai,India,2026-03-01 00:00:00,30.9,84,19.6,1009.6,Foggy,9.5,STN_MUM_02,WeatherPro,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Delhi,India,2026-03-01 00:00:00,38.9,52,27.9,996.4,Partly Cloudy,3.2,STN_DEL_01,AtmosAPI,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Pune,India,2026-03-01 00:00:00,33.6,60,9.4,944.9,Partly Cloudy,0.8,STN_PNE_01,ClimaData,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Chennai,India,2026-03-01 01:00:00,36.0,64,17.6,1009.5,Clear,14.9,STN_CHN_02,WeatherPro,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Bangalore,India,2026-03-01 01:00:00,23.6,59,14.2,908.2,Partly Cloudy,9.2,STN_BLR_01,SkyWatch,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Hyderabad,India,2026-03-01 01:00:00,31.7,64,18.8,994.8,Cloudy,3.5,STN_HYD_01,ClimaData,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw
Mumbai,India,2026-03-01 01:00:00,27.3,86,13.9,1008.6,Haze,15.2,STN_MUM_01,WeatherPro,null,null,null,null,null,null,2026-04-20T03:20:49.142Z,2026-04-20,/Volumes/Weather_Analytics/Data/raw_weather_data/weather_01.csv,2026-04-08 05:53:11,raw


In [0]:
spark.read.table("Weather_Analytics.bronze.bronze_station_master").show()

+----------+---------+------------+---------+------------+
|station_id|     city|station_zone|is_active|commissioned|
+----------+---------+------------+---------+------------+
|STN_BLR_01|Bangalore|       South|     true|  2024-01-01|
|STN_BLR_02|Bangalore|       South|     true|  2024-01-01|
|STN_CHN_01|  Chennai|       South|     true|  2024-01-01|
|STN_CHN_02|  Chennai|       South|     true|  2024-01-01|
|STN_DEL_01|    Delhi|       North|     true|  2024-01-01|
|STN_DEL_02|    Delhi|       North|     true|  2024-01-01|
|STN_HYD_01|Hyderabad|       South|     true|  2024-01-01|
|STN_HYD_02|Hyderabad|       South|     true|  2024-01-01|
|STN_MUM_01|   Mumbai|        West|     true|  2024-01-01|
|STN_MUM_02|   Mumbai|        West|     true|  2024-01-01|
|STN_PNE_01|     Pune|        West|     true|  2024-01-01|
|STN_PNE_02|     Pune|        West|     true|  2024-01-01|
+----------+---------+------------+---------+------------+



In [0]:
df = spark.read.table("Weather_Analytics.bronze.bronze_weather")
df.printSchema()

root
 |-- city: string (nullable = true)
 |-- country: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- temperature: string (nullable = true)
 |-- humidity: string (nullable = true)
 |-- wind_speed: string (nullable = true)
 |-- pressure: string (nullable = true)
 |-- weather_condition: string (nullable = true)
 |-- rainfall: string (nullable = true)
 |-- station_id: string (nullable = true)
 |-- data_provider: string (nullable = true)
 |-- uv_index: string (nullable = true)
 |-- visibility: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- feels_like_temperature: string (nullable = true)
 |-- sensor_payload: string (nullable = true)
 |-- _rescued_data: string (nullable = true)
 |-- ingestion_time: timestamp (nullable = true)
 |-- load_date: date (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)
 |-- record_status: string (nullable = true)



In [0]:
spark.sql("describe history Weather_Analytics.bronze.bronze_weather").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
14,2026-04-20T03:21:30.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 12, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,10,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 207, numOutputRows -> 72, numOutputBytes -> 8472, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
13,2026-04-20T03:21:29.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 11, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,9,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 106, numOutputRows -> 72, numOutputBytes -> 7322, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
12,2026-04-20T03:21:27.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 10, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,8,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 201, numOutputRows -> 72, numOutputBytes -> 7915, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
11,2026-04-20T03:21:25.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 9, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,7,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 205, numOutputRows -> 288, numOutputBytes -> 9729, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
10,2026-04-20T03:21:21.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 8, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,6,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 194, numOutputRows -> 144, numOutputBytes -> 9563, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
9,2026-04-20T03:21:19.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 7, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,6,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 84, numOutputRows -> 72, numOutputBytes -> 7463, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
8,2026-04-20T03:21:16.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 6, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,5,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflictDetectionTimeMs -> 119, numOutputRows -> 72, numOutputBytes -> 7249, numAddedFiles -> 1)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13
7,2026-04-20T03:21:14.000Z,70776481240661,hariharan110704@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 4f232b87-d13f-4324-b6b2-358be3c5f83f, epochId -> 5, statsOnLoad -> true)",null,List(3286725678381224),06d9de0b-b4db-4395-96de-2ef81c21574c,0420-031833-a8vbp5r3-v2n,4,WriteSerializable,true,"Map(numRemovedFiles -> 0, conflic

In [0]:
#  %sql
#  drop table Weather_Analytics.bronze.bronze_weather


In [0]:
# dbutils.fs.rm("/Volumes/Weather_Analytics/bronze/weather_data_checkpoint", True)
# dbutils.fs.rm("/Volumes/Weather_Analytics/Data/schema", True)

In [0]:
"""weather_data=spark.readStream.format("cloudFiles") \
  .option("cloudFiles.format", "csv") \
  .option("header", "true")\
  .option("cloudFiles.schemaLocation", "/Volumes/weather_analytics/data/schema") \
  .option("cloudFiles.schemaEvolutionMode", "addNewColumns") \
  .option("cloudFiles.maxFilesPerTrigger", "1") \
  .option("cloudFiles.inferColumnTypes", "true") \
  .load("/Volumes/weather_analytics/data/raw_weather_data/")
  .withcolumn("ingestion_timestamp", F.current_timestamp()) \
  .withColumn("load_date", F.current_date())\
  .withColumn("source_file", F.col("_metadata.file_path")) \
  .withColumn("batch_id", F.col("_metadata.file_modification_time").cast("string"))\
  .withColumn("batch_timestamp", F.col("_metadata.batchTimestamp")) \
  .withColumn("record_status", F.lit("valid")) \
  .withColumn("file_modification_time", F.col("_metadata.file_modification_time")) \
"""
  

'weather_data=spark.readStream.format("cloudFiles")   .option("cloudFiles.format", "csv")   .option("header", "true")  .option("cloudFiles.schemaLocation", "/Volumes/weather_analytics/data/schema")   .option("cloudFiles.schemaEvolutionMode", "addNewColumns")   .option("cloudFiles.maxFilesPerTrigger", "1")   .option("cloudFiles.inferColumnTypes", "true")   .load("/Volumes/weather_analytics/data/raw_weather_data/")\n  .withcolumn("ingestion_timestamp", F.current_timestamp())   .withColumn("load_date", F.current_date())  .withColumn("source_file", F.col("_metadata.file_path"))   .withColumn("batch_id", F.col("_metadata.file_modification_time").cast("string"))  .withColumn("batch_timestamp", F.col("_metadata.batchTimestamp"))   .withColumn("record_status", F.lit("valid"))   .withColumn("file_modification_time", F.col("_metadata.file_modification_time")) '

In [0]:
"""weather_data.writeStream \
  .format("delta") \
  .option("mergeSchema","true") \
  .outputMode("append") \
  .option("checkpointLocation", "/Volumes/weather_analytics/bronze/weather_data_checkpoint") \
  .trigger(once=True) \
  .table("weather_analytics.bronze.bronze_weather_data")"""

'weather_data.writeStream   .format("delta")   .option("mergeSchema","true")   .outputMode("append")   .option("checkpointLocation", "/Volumes/weather_analytics/bronze/weather_data_checkpoint")   .trigger(once=True)   .table("weather_analytics.bronze.bronze_weather_data")'

In [0]:
# from pyspark.sql import functions as F
# from pyspark.sql.types import *

# required_cols = [
#     "sensor_payload",
#     "temperature",
#     "humidity",
#     "wind_speed",
#     "pressure",
#     "rainfall",
#     "uv_index",
#     "visibility",
#     "weather_condition"
# ]

# for col_name in required_cols:
#     if col_name not in weather_data.columns:
#         weather_data = weather_data.withColumn(col_name, F.lit(None).cast("string"))

# # -------------------------
# # JSON schema
# # -------------------------
# json_schema = StructType([
#     StructField("temperature", DoubleType(), True),
#     StructField("humidity", DoubleType(), True),
#     StructField("wind_speed", DoubleType(), True),
#     StructField("pressure", DoubleType(), True),
#     StructField("weather_condition", StringType(), True),
#     StructField("rainfall", DoubleType(), True),
#     StructField("uv_index", DoubleType(), True),
#     StructField("visibility", DoubleType(), True)
# ])

# columns = weather_data.columns

# if "sensor_payload" in columns:
#     payload_col = F.col("sensor_payload")
# else:
#     payload_col = F.lit(None)


# weather_data = weather_data.withColumn(
#     "parsed_json",
#     F.from_json(payload_col, json_schema)
# )

# # -------------------------
# # Merge CSV + JSON 
# # -------------------------
# weather_data = weather_data \
#     .withColumn("temperature", F.coalesce(F.col("temperature").cast("string"), F.col("parsed_json.temperature").cast("string"))) \
#     .withColumn("humidity", F.coalesce(F.col("humidity").cast("string"), F.col("parsed_json.humidity").cast("string"))) \
#     .withColumn("wind_speed", F.coalesce(F.col("wind_speed").cast("string"), F.col("parsed_json.wind_speed").cast("string"))) \
#     .withColumn("pressure", F.coalesce(F.col("pressure").cast("string"), F.col("parsed_json.pressure").cast("string"))) \
#     .withColumn("rainfall", F.coalesce(F.col("rainfall").cast("string"), F.col("parsed_json.rainfall").cast("string"))) \
#     .withColumn("weather_condition", F.coalesce(F.col("weather_condition"), F.col("parsed_json.weather_condition"))) \
#     .withColumn("uv_index", F.coalesce(F.col("uv_index").cast("string"), F.col("parsed_json.uv_index").cast("string"))) \
#     .withColumn("visibility", F.coalesce(F.col("visibility").cast("string"), F.col("parsed_json.visibility").cast("string"))) \
#     .drop("parsed_json")
# # -------------------------
# # SAFE CAST
# # -------------------------
# weather_data = weather_data \
#     .withColumn("temperature", F.expr("try_cast(temperature as double)")) \
#     .withColumn("humidity", F.expr("try_cast(humidity as double)")) \
#     .withColumn("wind_speed", F.expr("try_cast(wind_speed as double)")) \
#     .withColumn("pressure", F.expr("try_cast(pressure as double)")) \
#     .withColumn("rainfall", F.expr("try_cast(rainfall as double)"))

# # -------------------------
# # METADATA
# # -------------------------
